In [1]:
import datetime
from dataclasses import dataclass
import json
from pathlib import Path
from statistics import mean, stdev

In [2]:
HPARAMS_RESULT_DIR = Path().cwd().parent / "data" / "hparams"


@dataclass
class HparamResult:
    """Dataclass for storing output of the hyperparameter search."""
    params_percent: float
    loss: str
    lr: float
    distance: str
    cell_size: int
    knn_update_rate: int
    epochs: int

    time: float
    vram: int
    mse: float
    psnr: float
    ssim: float

    std_mse: float
    std_psnr: float
    std_ssim: float


def load_hparam_result(path: Path) -> list[HparamResult]:
    if not path.exists():
        raise ValueError(f"File {path} does not exist")

    results: list[HparamResult] = []

    with open(path, "r") as f:
        content = json.load(f)

        for epoch in [100, 200, 300, 500, 1000]:
            results.append(
                HparamResult(
                    params_percent=content["hparams"]["params_percent"],
                    loss=content["hparams"]["loss"],
                    lr=content["hparams"]["lr"],
                    distance=content["hparams"]["distance"],
                    cell_size=content["hparams"]["cell_size"],
                    knn_update_rate=content["hparams"]["knn_update_rate"],
                    epochs=epoch,

                    time=mean([content["statistics"][key][f"t_{epoch}_full"] for key in content["statistics"].keys()]),
                    vram=mean([content["statistics"][key][f"vram_{epoch}"] for key in content["statistics"].keys()]),
                    mse=mean([content["statistics"][key][f"mse_{epoch}"] for key in content["statistics"].keys()]),
                    psnr=mean([content["statistics"][key][f"psnr_{epoch}"] for key in content["statistics"].keys()]),
                    ssim=mean([content["statistics"][key][f"ssim_{epoch}"] for key in content["statistics"].keys()]),

                    std_mse=stdev([content["statistics"][key][f"mse_{epoch}"] for key in content["statistics"].keys()]),
                    std_psnr=stdev(
                        [content["statistics"][key][f"psnr_{epoch}"] for key in content["statistics"].keys()]),
                    std_ssim=stdev(
                        [content["statistics"][key][f"ssim_{epoch}"] for key in content["statistics"].keys()]),
                ))

    return results


def load_results() -> list[HparamResult]:
    results: list[HparamResult] = []

    for path in list(HPARAMS_RESULT_DIR.glob("*.json")):
        results.extend(load_hparam_result(path))

    return results


results = load_results()


In [3]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots



In [4]:
metrics = [
    ("mse", "std_mse", "MSE"),
    ("psnr", "std_psnr", "PSNR"),
    ("ssim", "std_ssim", "SSIM"),
]

rng = np.random.default_rng(0)

epoch_values = np.array(sorted({r.epochs for r in results}))
gap = np.diff(epoch_values).min() if len(epoch_values) > 1 else 1.0
half_width = 0.15 * gap
epochs = np.array([r.epochs for r in results], dtype=float)
x = epochs + rng.uniform(-half_width, half_width, len(results))


def to_seconds(t):
    if isinstance(t, (int, float)):
        return float(t)

    return t.hour * 3600 + t.minute * 60 + t.second + t.microsecond / 1e6


def fmt_hms(s):
    s = int(round(s))

    return f"{s // 3600}:{s % 3600 // 60:02d}:{s % 60:02d}"


losses = sorted({r.loss for r in results})
palette = px.colors.qualitative.Plotly
loss_color = {l: palette[i % len(palette)] for i, l in enumerate(losses)}
loss_arr = np.array([r.loss for r in results])

vram = np.array([r.vram for r in results], dtype=float)
max_px = 8
sizeref = 2 * vram.max() / max_px ** 2

base = np.array([
    [r.epochs, r.params_percent, r.loss, r.lr, r.distance,
     r.cell_size, r.knn_update_rate, fmt_hms(to_seconds(r.time)), r.vram]
    for r in results
], dtype=object)

hover_base = (
    "epochs: %{customdata[0]}<br>"
    "params: %{customdata[1]}%  ·  loss: %{customdata[2]}<br>"
    "lr: %{customdata[3]:.1e}  ·  distance: %{customdata[4]}<br>"
    "cell_size: %{customdata[5]}  ·  knn_rate: %{customdata[6]}<br>"
    "time: %{customdata[7]}  ·  vram: %{customdata[8]}<br>"
)

fig = make_subplots(rows=1, cols=len(metrics), horizontal_spacing=0.07)

for col, (attr, std_attr, label) in enumerate(metrics, start=1):
    y = np.array([getattr(r, attr) for r in results])
    std = np.array([getattr(r, std_attr) for r in results])
    custom = np.column_stack([base, std])

    for loss in losses:
        m = loss_arr == loss
        fig.add_trace(
            go.Scatter(
                x=x[m],
                y=y[m],
                mode="markers",
                name=loss,
                legendgroup=loss,
                showlegend=(col == 1),
                marker=dict(
                    color=loss_color[loss],
                    size=vram[m],
                    sizemode="area",
                    sizeref=sizeref,
                    sizemin=3,
                    opacity=0.75,
                    line=dict(width=0.5, color="white"),
                ),
                customdata=custom[m],
                hovertemplate=hover_base + f"{label}: %{{y:.4g}} ± %{{customdata[9]:.2g}}<extra></extra>",
            ),
            row=1, col=col,
        )
    fig.update_yaxes(title_text=label, row=1, col=col)

fig.update_layout(
    template="plotly_white",
    legend_title_text="Loss Function",
    height=450,
    width=1800,
)

fig.show()

In [5]:
def plot_colored_by(attr_name: str, legend_title: str):
    """Same plot as above, but colored by an arbitrary hparam instead of the loss function."""
    group_arr = np.array([getattr(r, attr_name) for r in results], dtype=object)
    groups = sorted(set(group_arr))
    group_color = {g: palette[i % len(palette)] for i, g in enumerate(groups)}

    fig = make_subplots(rows=1, cols=len(metrics), horizontal_spacing=0.07)

    for col, (attr, std_attr, label) in enumerate(metrics, start=1):
        y = np.array([getattr(r, attr) for r in results])
        std = np.array([getattr(r, std_attr) for r in results])
        custom = np.column_stack([base, std])

        for g in groups:
            m = group_arr == g
            fig.add_trace(
                go.Scatter(
                    x=x[m],
                    y=y[m],
                    mode="markers",
                    name=str(g),
                    legendgroup=str(g),
                    showlegend=(col == 1),
                    marker=dict(
                        color=group_color[g],
                        size=vram[m],
                        sizemode="area",
                        sizeref=sizeref,
                        sizemin=2,
                        opacity=0.75,
                        line=dict(width=0.5, color="white"),
                    ),
                    customdata=custom[m],
                    hovertemplate=hover_base + f"{label}: %{{y:.4g}} ± %{{customdata[9]:.2g}}<extra></extra>",
                ),
                row=1, col=col,
            )
        fig.update_yaxes(title_text=label, row=1, col=col)
        fig.update_xaxes(title_text="Epochs", row=1, col=col)

    fig.update_layout(
        template="plotly_white",
        legend_title_text=legend_title,
        height=450,
        width=1800,
    )

    fig.show()

In [6]:
plot_colored_by("lr", "Learning Rate")

In [7]:
plot_colored_by("distance", "Distance Metric")

In [8]:
plot_colored_by("cell_size", "Cell Size")

In [9]:
plot_colored_by("knn_update_rate", "KNN Update Rate")

In [10]:
plot_colored_by("params_percent", "Compression to %")